# T455 — Two clocks and geographic polar motion

## tl;dr

The exact atomic/Earth-rotation clock relation remains almost exactly on the ARA ridge. The geographic-pole traversal child recovers a stable approximately annual wave at 30- and 90-day grains. The full frozen predictive claim passes only 3/6 gates. A Di-ARA-only child helps longer-horizon forecasts, but a one-year-shifted child performs similarly, so most of that structure belongs to an annual parent carrier.

## Context & methods

**Who:** Earth. **What:** SI atomic day versus observed Earth-rotation day, plus geographic polar motion. **When:** daily IERS EOP C04 observations from 1984-01-01 through 2026-07-29. **Where:** parent two-clock ridge → geographic-pole amount/traversal child. **Why:** ask whether the same relational child supplies timing information at 1, 7, 30 and 90 days. **How:** frozen chronological splits, exact 0–2 clock relation, typed pole Irrationality Di-ARA, causal forecasts, false-time controls and moving-block bootstrap. The same-season audit is explicitly post-result.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\irrationality_di_ara\T455_two_clock_polar_motion')
RESULTS = ROOT / 'results'
windows = pd.read_csv(RESULTS/'T455_SCALE_WINDOWS.csv', parse_dates=['start_date','end_date'])
metrics = pd.read_csv(RESULTS/'T455_FORECAST_METRICS.csv')
controls = pd.read_csv(RESULTS/'T455_FALSE_TIME_CONTROLS.csv')
geometry = pd.read_csv(RESULTS/'T455_SCALE_GEOMETRY.csv')
quadrants = pd.read_csv(RESULTS/'T455_QUADRANT_OCCUPANCY.csv')
seasonal = pd.read_csv(RESULTS/'T455_POSTHOC_SEASONAL_AUDIT.csv')
result = json.loads((RESULTS/'T455_RESULT.json').read_text())
print(result)

## Data

The exact clock coordinate is shown without rescaling; the magnified panel reports only its nanounit distance from the ARA ridge.

In [2]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
daily = windows[windows.scale_days.eq(1)]
ax[0].plot(daily.end_date, daily.clock_ara, lw=.7, color='#4ea3ff')
ax[0].axhline(1, color='white', ls='--', lw=1)
ax[0].set_ylabel('Exact two-clock ARA (0–2)')
ax[0].set_title('Earth-clock relation remains on the ARA ridge')
ax[1].plot(daily.end_date, daily.clock_ridge_nano, lw=.7, color='#ff9f43')
ax[1].axhline(0, color='white', ls='--', lw=1)
ax[1].set_ylabel('(ARA − 1) × 10⁹')
ax[1].set_xlabel('Observation date')
plt.tight_layout(); plt.show()

## Results

The child geometry becomes more coherent as the observational grain grows. This is a scale relation, not a universal landmark forced at every grain.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for scale, color in [(30, '#4ea3ff'), (90, '#ff9f43')]:
    d = windows[windows.scale_days.eq(scale)].dropna(subset=['pole_amount_ara','pole_traversal_ara'])
    axes[0].scatter(d.pole_amount_ara, d.pole_traversal_ara, s=11, alpha=.45, label=f'{scale}-day')
axes[0].axvline(1, color='grey', ls='--'); axes[0].axhline(1, color='grey', ls='--')
axes[0].set(xlabel='Pole amount ARA (0–2)', ylabel='Pole traversal ARA (0–2)', title='Coarse-grain Irrationality Di-ARA')
axes[0].legend()
for split, g in geometry.groupby('split'):
    axes[1].plot(g.scale_days, g.median_traversal_ara, marker='o', label=split)
axes[1].axhline(1, color='grey', ls='--')
axes[1].set(xscale='log', xlabel='Grain (days, log scale)', ylabel='Median traversal ARA', title='Traversal orientation by grain')
axes[1].legend()
plt.tight_layout(); plt.show()

In [4]:
hold = metrics[metrics.split.eq('holdout')]
clock = hold[hold.model.eq('clock_only')][['scale_days','horizon_windows','mae']].rename(columns={'mae':'clock_mae'})
cand = hold[hold.model.isin(['clock_pole_diara','full_child'])].merge(clock, on=['scale_days','horizon_windows'])
cand['improvement_pct'] = 100*(cand.clock_mae-cand.mae)/cand.clock_mae
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for (model, horizon), g in cand.groupby(['model','horizon_windows']):
    axes[0].plot(g.scale_days, g.improvement_pct, marker='o', label=f'{model}, h={horizon}')
axes[0].axhline(0, color='grey', ls='--'); axes[0].set_xscale('log')
axes[0].set(xlabel='Grain (days)', ylabel='Holdout MAE improvement over clock-only (%)', title='Frozen prospective result')
axes[0].legend(fontsize=8)
c = controls[(controls.candidate_model.eq('clock_pole_diara')) & controls.horizon_windows.eq(4)]
for name, g in c.groupby('control'):
    axes[1].plot(g.scale_days, g.improvement_vs_clock_pct, marker='o', label=name)
axes[1].axhline(0, color='grey', ls='--'); axes[1].set_xscale('log')
axes[1].set(xlabel='Grain (days)', ylabel='Improvement over clock-only (%)', title='Four-window false-time controls')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [5]:
fig, ax = plt.subplots(figsize=(10, 5))
for horizon, g in seasonal.groupby('horizon_windows'):
    ax.plot(g.scale_days, g.diara_improvement_over_season_pct, marker='o', label=f'{horizon} window(s)')
ax.axhline(0, color='grey', ls='--'); ax.set_xscale('log')
ax.set(xlabel='Grain (days)', ylabel='Live Di-ARA improvement over same-season baseline (%)', title='Post-result seasonal-parent diagnostic')
ax.legend(); plt.tight_layout(); plt.show()

## Takeaways

1. The exact two-clock parent is a genuine ridge-level relation.
2. The pole child reconstructs an annual directional wave at 30–90 day grains.
3. The full frozen prospective claim does not transfer across grains.
4. The relational Di-ARA child is more useful than raw pole position, especially over longer horizons.
5. A one-year-shifted control carries nearly the same broad signal, so the annual parent must be removed before claiming a live timing handover.
6. The next confirmation should freeze live-minus-same-season prediction and preserve signed traversal orientation.